# Eval Trace Analyzer
Analysis of `eval_game.py` results — prompt length, response length, thinking blocks, truncation, and latency.

In [67]:
import json, re, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

In [ ]:
# ── File loading ──────────────────────────────────────────────────────────────
RESULTS_DIR = pathlib.Path('results')

files = sorted(RESULTS_DIR.glob('eval_trace_*.json'))
print(f'Files found: {len(files)}')
for f in files:
    print(f'  {f.name}')

# Use the last file by default; replace with a specific path if needed
TRACE_FILE = files[4]
print(f'\nAnalysing: {TRACE_FILE.name}')

In [ ]:
# ── Parsing ───────────────────────────────────────────────────────────────────
_THINK_RE = re.compile(r'<think>(.*?)</think>', re.DOTALL)

_CALL_COLS = [
    'move_index', 'move_san', 'call_num', 'is_retry', 'elapsed_s',
    'max_tokens', 'finish_reason', 'prompt_chars', 'raw_chars',
    'think_chars', 'stripped_chars', 'is_think_only', 'truncated',
]

def extract_think_chars(raw: str) -> int:
    m = _THINK_RE.search(raw)
    return len(m.group(0)) if m else 0

def section_chars(sections: list[dict]) -> dict[str, int]:
    """Character count per prompt section label."""
    return {s['label']: len(s.get('content', '')) for s in sections}

def parse_trace(path: pathlib.Path):
    data = json.loads(path.read_text())
    meta = {
        'game':           data.get('game', {}),
        'config':         data.get('config'),
        'run_config':     data.get('run_config', {}),
        'run_config_file':data.get('run_config_file'),
        'max_tokens':     data.get('max_tokens'),
        'analysis_depth': data.get('analysis_depth'),
        'fixture':        data.get('fixture'),
    }

    move_rows, call_rows, section_rows = [], [], []

    for t in data['traces']:
        if t.get('san') is None:
            continue

        calls     = t.get('lm_calls', [])
        n_calls   = len(calls)
        total_el  = sum(c.get('elapsed_s', 0) for c in calls)
        trunc_raw = any(c.get('finish_reason') == 'length' for c in calls)

        def _stripped_truncated(c):
            if c.get('finish_reason') != 'length':
                return False
            think = extract_think_chars(c.get('raw_response', ''))
            raw   = c.get('response_chars', 0)
            return (raw - think) >= raw * 0.5

        trunc_stripped = any(_stripped_truncated(c) for c in calls)

        sec = section_chars(t.get('prompt_sections', []))
        for label, chars in sec.items():
            section_rows.append({
                'move_index': t['index'],
                'san':        t.get('san'),
                'label':      label,
                'chars':      chars,
            })

        move_rows.append({
            'index':              t['index'],
            'move_number':        t.get('move_number'),
            'san':                t.get('san'),
            'color':              t.get('color'),
            'quality':            t.get('quality'),
            'question_type':      t.get('question_type'),
            'eval_cp':            t.get('eval_cp'),
            'eval_loss_cp':       t.get('eval_loss_cp'),
            'prompt_chars':       t.get('prompt_chars', 0),
            'prompt_tokens':      t.get('prompt_tokens_est', 0),
            'commentary_chars':   t.get('commentary_chars', 0),
            'commentary_empty':   t.get('commentary_empty', False),
            'retried':            t.get('retried', False),
            'n_calls':            n_calls,
            'total_elapsed_s':    total_el,
            'truncated_raw':      trunc_raw,
            'truncated_stripped': trunc_stripped,
        })

        for ci, c in enumerate(calls):
            raw       = c.get('raw_response', '')
            think_ch  = extract_think_chars(raw)
            raw_ch    = c.get('response_chars', len(raw))
            strip_ch  = c.get('stripped_chars', 0)
            call_rows.append({
                'move_index':    t['index'],
                'move_san':      t.get('san'),
                'call_num':      ci,
                'is_retry':      ci > 0,
                'elapsed_s':     c.get('elapsed_s', 0),
                'max_tokens':    c.get('max_tokens'),
                'finish_reason': c.get('finish_reason'),
                'prompt_chars':  c.get('prompt_chars', 0),
                'raw_chars':     raw_ch,
                'think_chars':   think_ch,
                'stripped_chars':strip_ch,
                'is_think_only': c.get('is_think_only', False),
                'truncated':     c.get('finish_reason') == 'length',
            })

    moves    = pd.DataFrame(move_rows)
    calls_df = pd.DataFrame(call_rows, columns=_CALL_COLS) if call_rows else pd.DataFrame(columns=_CALL_COLS)
    secs_df  = pd.DataFrame(section_rows)
    return meta, moves, calls_df, secs_df


meta, moves, calls, secs = parse_trace(TRACE_FILE)
g = meta['game']
print(f"Game   : {g.get('White')} vs {g.get('Black')} ({g.get('Date')})")
print(f"Config : {meta['config']}  |  max_tokens={meta['max_tokens']}")
if meta.get('run_config_file'):
    print(f"JSON cfg: {pathlib.Path(meta['run_config_file']).name}")
print(f"Moves  : {len(moves)}  |  LLM calls: {len(calls)}")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
first_calls = calls[calls['call_num'] == 0]

summary = {
    'Total moves':                          len(moves),
    'Total LLM calls':                      len(calls),
    'Retries':                              int(calls['is_retry'].sum()),
    'Empty responses':                      int(moves['commentary_empty'].sum()),
    'Think-only calls':                     int(calls['is_think_only'].sum()),

    'Avg prompt length, chars':             round(first_calls['prompt_chars'].mean(), 1),
    'Median prompt length, chars':          round(first_calls['prompt_chars'].median(), 1),
    'Max prompt length, chars':             int(first_calls['prompt_chars'].max()),

    'Avg response length (stripped), chars':round(first_calls['stripped_chars'].mean(), 1),
    'Avg thinking length, chars':           round(first_calls['think_chars'].mean(), 1),
    'Avg raw length (with thinking), chars':round(first_calls['raw_chars'].mean(), 1),

    'Truncated (incl. thinking)':           int(calls['truncated'].sum()),
    'Truncated response (excl. thinking)':  int(sum(
        1 for _, r in calls.iterrows()
        if r['truncated'] and r['stripped_chars'] > r['think_chars'] * 0.5
    )),

    'Avg latency, s':                       round(first_calls['elapsed_s'].mean(), 2),
    'Median latency, s':                    round(first_calls['elapsed_s'].median(), 2),
    'Max latency, s':                       round(first_calls['elapsed_s'].max(), 2),
}

pd.DataFrame(summary.items(), columns=['Metric', 'Value'])

## Prompt Length

In [ ]:
# ── Prompt length distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Prompt Length Distribution', fontsize=13, fontweight='bold')

pc = first_calls['prompt_chars'].dropna()

# Histogram
ax = axes[0]
ax.hist(pc, bins=25, color='#2196F3', alpha=0.85, edgecolor='white')
ax.axvline(pc.mean(),            color='red',    linestyle='--', lw=1.5, label=f'mean={pc.mean():.0f}')
ax.axvline(pc.median(),          color='orange', linestyle=':',  lw=1.5, label=f'med={pc.median():.0f}')
ax.axvline(pc.quantile(0.95),    color='purple', linestyle='-.', lw=1.2, label=f'p95={pc.quantile(0.95):.0f}')
ax.set_title('Histogram')
ax.set_xlabel('chars')
ax.set_ylabel('calls')
ax.legend(fontsize=8)

# By move color
ax = axes[1]
for color, hue in [('white', '#4C72B0'), ('black', '#C44E52')]:
    subset = moves.loc[moves['color'] == color, 'prompt_chars'].dropna()
    ax.hist(subset, bins=20, alpha=0.6, color=hue, edgecolor='white', label=color.capitalize())
ax.set_title('White vs Black')
ax.set_xlabel('chars')
ax.set_ylabel('count')
ax.legend()

# Cumulative
ax = axes[2]
sorted_pc = np.sort(pc)
cdf = np.arange(1, len(sorted_pc) + 1) / len(sorted_pc)
ax.plot(sorted_pc, cdf, color='#2196F3', lw=2)
ax.axvline(pc.quantile(0.5),  color='orange', linestyle=':', lw=1.2, alpha=0.8, label='p50')
ax.axvline(pc.quantile(0.9),  color='red',    linestyle=':', lw=1.2, alpha=0.8, label='p90')
ax.axvline(pc.quantile(0.95), color='purple', linestyle=':', lw=1.2, alpha=0.8, label='p95')
ax.set_title('CDF')
ax.set_xlabel('chars')
ax.set_ylabel('fraction of calls')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Per-section contribution to prompt length ─────────────────────────────────
if not secs.empty:
    sec_mean  = secs.groupby('label')['chars'].mean().sort_values(ascending=False)
    sec_total = secs.groupby('label')['chars'].sum().sort_values(ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle('Prompt Section Contributions', fontsize=13, fontweight='bold')

    colors = plt.cm.Set2(np.linspace(0, 1, len(sec_mean)))

    # Mean size per section
    ax = axes[0]
    bars = ax.barh(sec_mean.index[::-1], sec_mean.values[::-1], color=colors[::-1], edgecolor='white')
    for bar, val in zip(bars, sec_mean.values[::-1]):
        ax.text(val + 5, bar.get_y() + bar.get_height()/2, f'{val:.0f}', va='center', fontsize=8)
    ax.set_title('Mean section length, chars')
    ax.set_xlabel('chars')

    # Stacked bar — per-move prompt composition
    ax = axes[1]
    pivot = secs.pivot_table(index='move_index', columns='label', values='chars', fill_value=0)
    bottom = np.zeros(len(pivot))
    col_colors = dict(zip(sec_mean.index, plt.cm.Set2(np.linspace(0, 1, len(sec_mean)))))
    for label in sec_mean.index:
        if label in pivot.columns:
            vals = pivot[label].values
            ax.bar(pivot.index, vals, bottom=bottom, label=label,
                   color=col_colors[label], alpha=0.85, edgecolor='none')
            bottom += vals
    ax.set_title('Prompt composition per move')
    ax.set_xlabel('move index')
    ax.set_ylabel('chars')
    ax.legend(fontsize=7, loc='upper right', ncol=2)

    plt.tight_layout()
    plt.show()

    df_sec = pd.DataFrame({
        'Section':          sec_mean.index,
        'Avg chars':        sec_mean.values.round(1),
        'Share of prompt':  (sec_mean / moves['prompt_chars'].mean() * 100).round(1),
    }).reset_index(drop=True)
    display(df_sec)
else:
    print('No prompt section data (older trace format).')

In [ ]:
# ── Prompt length over move sequence ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 3.5))
fig.suptitle('Prompt Length per Move', fontsize=13, fontweight='bold')

for _, row in moves.iterrows():
    c = '#f0f8ff' if row['color'] == 'white' else '#fff5f0'
    ax.axvspan(row['index'] - 0.5, row['index'] + 0.5, alpha=0.25, color=c, linewidth=0)

ax.plot(moves['index'], moves['prompt_chars'], color='#2196F3', lw=1.5, marker='o', markersize=2.5)
ax.axhline(moves['prompt_chars'].mean(), color='red', linestyle='--', lw=1, alpha=0.7,
           label=f"mean={moves['prompt_chars'].mean():.0f}")

threshold = moves['prompt_chars'].mean() + 1.5 * moves['prompt_chars'].std()
outliers = moves[moves['prompt_chars'] > threshold]
ax.scatter(outliers['index'], outliers['prompt_chars'],
           color='red', s=40, zorder=5, label='outlier (>mean+1.5σ)')

ax.set_xlabel('move index')
ax.set_ylabel('chars')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Response Length, Thinking, and Latency

In [ ]:
# ── Response length distributions ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Response Length Distributions (First Calls)', fontsize=13, fontweight='bold')

for ax, (col, label, color) in zip(axes, [
    ('stripped_chars', 'Response (excl. thinking)', '#4C72B0'),
    ('think_chars',    'Thinking block',             '#DD8452'),
    ('raw_chars',      'Raw (incl. thinking)',        '#55A868'),
]):
    data = first_calls[col].dropna()
    ax.hist(data, bins=20, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(data.mean(),   color='red',    linestyle='--', lw=1.5, label=f'mean={data.mean():.0f}')
    ax.axvline(data.median(), color='orange', linestyle=':',  lw=1.5, label=f'med={data.median():.0f}')
    ax.set_title(label)
    ax.set_xlabel('chars')
    ax.set_ylabel('calls')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Latency distribution ───────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('LLM Latency', fontsize=13, fontweight='bold')

elapsed = first_calls['elapsed_s'].dropna()
ax1.hist(elapsed, bins=25, color='#8172B2', alpha=0.85, edgecolor='white')
ax1.axvline(elapsed.mean(),           color='red',    linestyle='--', lw=1.5, label=f'mean={elapsed.mean():.2f}s')
ax1.axvline(elapsed.median(),         color='orange', linestyle=':',  lw=1.5, label=f'med={elapsed.median():.2f}s')
ax1.axvline(elapsed.quantile(0.95),   color='purple', linestyle='-.', lw=1.2, label=f'p95={elapsed.quantile(0.95):.2f}s')
ax1.set_title('Histogram')
ax1.set_xlabel('seconds')
ax1.set_ylabel('calls')
ax1.legend(fontsize=9)

groups = [first_calls.loc[first_calls['is_retry'] == v, 'elapsed_s'].dropna() for v in [False, True]]
ax2.boxplot(groups, labels=['First call', 'Retry'], patch_artist=True,
            boxprops=dict(facecolor='#8172B2', alpha=0.6))
ax2.set_title('First Call vs Retry')
ax2.set_ylabel('seconds')

plt.tight_layout()
plt.show()

In [ ]:
# ── Dynamics over game moves ───────────────────────────────────────────────────
fc = first_calls[['move_index', 'stripped_chars', 'think_chars', 'raw_chars', 'elapsed_s', 'truncated']].copy()
fc = fc.rename(columns={'move_index': 'index'})
mfc = moves.merge(fc, on='index', how='left')

fig, axes = plt.subplots(4, 1, figsize=(16, 13), sharex=True)
fig.suptitle('Dynamics over Game Moves', fontsize=13, fontweight='bold')

x = mfc['index']
for ax in axes:
    for _, row in mfc.iterrows():
        c = '#f0f8ff' if row['color'] == 'white' else '#fff5f0'
        ax.axvspan(row['index'] - 0.5, row['index'] + 0.5, alpha=0.25, color=c, linewidth=0)

# Prompt
axes[0].plot(x, mfc['prompt_chars'], color='#2196F3', lw=1.5, marker='o', markersize=2)
axes[0].axhline(mfc['prompt_chars'].mean(), color='red', linestyle='--', lw=1, alpha=0.6,
                label=f"mean={mfc['prompt_chars'].mean():.0f}")
axes[0].set_ylabel('chars')
axes[0].set_title('Prompt Length')
axes[0].legend(fontsize=8)

# Response + thinking
axes[1].bar(x, mfc['stripped_chars'], color='#4C72B0', alpha=0.7, label='response')
axes[1].bar(x, mfc['think_chars'],    color='#DD8452', alpha=0.7, label='thinking', bottom=mfc['stripped_chars'])
trunc_idx = mfc[mfc['truncated_raw'] == True]['index']
if len(trunc_idx):
    trunc_h = [mfc.loc[mfc['index'] == i, 'raw_chars'].values[0] + 20 for i in trunc_idx]
    axes[1].scatter(trunc_idx, trunc_h, marker='v', color='red', s=60, zorder=5, label='truncated')
axes[1].set_ylabel('chars')
axes[1].set_title('Response + Thinking')
axes[1].legend(fontsize=8)

# Latency
axes[2].plot(x, mfc['total_elapsed_s'], color='#8172B2', marker='o', markersize=3, lw=1.5)
axes[2].axhline(mfc['total_elapsed_s'].mean(), color='red', linestyle='--', lw=1, alpha=0.6,
                label=f"mean={mfc['total_elapsed_s'].mean():.2f}s")
axes[2].set_ylabel('seconds')
axes[2].set_title('Latency')
axes[2].legend(fontsize=8)

# Eval CP
has_cp = mfc['eval_cp'].notna()
axes[3].plot(mfc.loc[has_cp, 'index'], mfc.loc[has_cp, 'eval_cp'],
             color='#2ca02c', lw=1.5)
axes[3].axhline(0, color='gray', lw=0.8)
axes[3].set_ylabel('centipawns')
axes[3].set_xlabel('move index')
axes[3].set_title("Position Evaluation (White's perspective)")

plt.tight_layout()
plt.show()

## Truncation and Move Quality

In [ ]:
# ── Truncation analysis ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Truncation Analysis (finish_reason=length)', fontsize=13, fontweight='bold')

trunc_counts = calls['truncated'].value_counts()
axes[0].pie(
    [trunc_counts.get(False, 0), trunc_counts.get(True, 0)],
    labels=['Not truncated', 'Truncated'],
    autopct='%1.1f%%', colors=['#55A868', '#C44E52'],
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}, startangle=90,
)
axes[0].set_title('All LLM Calls')

colors_sc = calls['truncated'].map({True: '#C44E52', False: '#4C72B0'})
axes[1].scatter(calls['think_chars'], calls['stripped_chars'],
                c=colors_sc, alpha=0.7, s=40, edgecolors='white', lw=0.5)
axes[1].set_xlabel('thinking, chars')
axes[1].set_ylabel('response (stripped), chars')
axes[1].set_title('Thinking vs Response')
axes[1].legend(handles=[
    Patch(facecolor='#C44E52', label='Truncated'),
    Patch(facecolor='#4C72B0', label='Not truncated'),
], fontsize=9)

plt.tight_layout()
plt.show()

trunc_answer = calls[calls['truncated'] & (calls['stripped_chars'] > calls['think_chars'] * 0.5)]
print(f"Truncated total: {trunc_counts.get(True, 0)} of {len(calls)}")
print(f"  of which response truncated (not thinking): {len(trunc_answer)}")
print(f"  of which thinking truncated:                {trunc_counts.get(True, 0) - len(trunc_answer)}")

In [ ]:
# ── Move quality and question type ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Move Quality and Question Type', fontsize=13, fontweight='bold')

QUALITY_ORDER  = ['book', 'good', 'inaccuracy', 'mistake', 'blunder']
QUALITY_COLORS = {'book':'#8dc6ff','good':'#55A868','inaccuracy':'#FFD166','mistake':'#f4a261','blunder':'#C44E52'}

q_counts = moves['quality'].value_counts()
q_order  = [q for q in QUALITY_ORDER if q in q_counts]
axes[0].bar(q_order, [q_counts[q] for q in q_order],
            color=[QUALITY_COLORS.get(q, '#aaa') for q in q_order], edgecolor='white')
axes[0].set_title('Move Quality Distribution')
axes[0].set_xlabel('quality')
axes[0].set_ylabel('count')
for i, q in enumerate(q_order):
    axes[0].text(i, q_counts[q] + 0.3, str(q_counts[q]), ha='center', fontsize=9)

qt = moves['question_type'].value_counts()
axes[1].barh(qt.index, qt.values, color='#4C72B0', alpha=0.8, edgecolor='white')
axes[1].set_title('Question Types Sent to LLM')
axes[1].set_xlabel('count')
for i, v in enumerate(qt.values):
    axes[1].text(v + 0.2, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Correlations

In [ ]:
# ── Correlations ───────────────────────────────────────────────────────────────
fc2 = first_calls[['move_index','prompt_chars','stripped_chars','think_chars','elapsed_s','truncated']].rename(columns={'move_index':'index'})
mfc2 = moves.merge(fc2, on='index', how='left')

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Correlations', fontsize=13, fontweight='bold')

pairs = [
    ('prompt_chars_x', 'stripped_chars', 'Prompt → Response',  '#4C72B0'),
    ('prompt_chars_x', 'think_chars',    'Prompt → Thinking',  '#DD8452'),
    ('prompt_chars_x', 'elapsed_s',      'Prompt → Latency',   '#55A868'),
    ('stripped_chars', 'elapsed_s',      'Response → Latency', '#8172B2'),
]

for ax, (xcol, ycol, title, color) in zip(axes, pairs):
    data = mfc2[[xcol, ycol]].dropna()
    ax.scatter(data[xcol], data[ycol], color=color, alpha=0.6, s=30, edgecolors='white', lw=0.4)
    if len(data) > 2:
        z  = np.polyfit(data[xcol], data[ycol], 1)
        xs = np.linspace(data[xcol].min(), data[xcol].max(), 100)
        ax.plot(xs, np.poly1d(z)(xs), color='red', lw=1.5, linestyle='--', alpha=0.8)
        r = data[[xcol, ycol]].corr().iloc[0, 1]
        ax.set_title(f'{title}  (r={r:.2f})')
    else:
        ax.set_title(title)
    ax.set_xlabel(xcol.replace('_x', ''))
    ax.set_ylabel(ycol)

plt.tight_layout()
plt.show()

## Detailed Move Table

In [ ]:
# ── Detailed per-move table ────────────────────────────────────────────────────
detail = mfc[[
    'move_number', 'san', 'color', 'quality', 'question_type',
    'prompt_chars', 'stripped_chars', 'think_chars', 'total_elapsed_s',
    'n_calls', 'retried', 'truncated_raw', 'truncated_stripped', 'commentary_empty'
]].copy().rename(columns={
    'move_number':        '#',
    'san':                'Move',
    'color':              'Color',
    'quality':            'Quality',
    'question_type':      'Question type',
    'prompt_chars':       'Prompt',
    'stripped_chars':     'Response',
    'think_chars':        'Thinking',
    'total_elapsed_s':    'Time, s',
    'n_calls':            'Calls',
    'retried':            'Retry',
    'truncated_raw':      'Trunc (raw)',
    'truncated_stripped': 'Trunc (resp)',
    'commentary_empty':   'Empty',
})

def _style(row):
    s = [''] * len(row)
    cols = list(row.index)
    if row.get('Empty'):       s[cols.index('Empty')]       = 'background-color: #ffcccc'
    if row.get('Trunc (raw)'): s[cols.index('Trunc (raw)')] = 'background-color: #ffe0b2'
    if row.get('Retry'):       s[cols.index('Retry')]       = 'background-color: #fff9c4'
    return s

detail.style\
    .apply(_style, axis=1)\
    .format({'Time, s': '{:.2f}', 'Prompt': '{:.0f}', 'Response': '{:.0f}', 'Thinking': '{:.0f}'})\
    .hide(axis='index')

## Multi-File Comparison

In [ ]:
# ── Multi-file comparison ─────────────────────────────────────────────────────
if len(files) > 1:
    records = []
    for f in files:
        m, mv, cl, _ = parse_trace(f)
        fc_ = cl[cl['call_num'] == 0]
        records.append({
            'file':          f.stem,
            'config':        m['config'],
            'max_tokens':    m['max_tokens'],
            'moves':         len(mv),
            'calls':         len(cl),
            'avg prompt':    round(fc_['prompt_chars'].mean(), 1),
            'avg response':  round(fc_['stripped_chars'].mean(), 1),
            'avg thinking':  round(fc_['think_chars'].mean(), 1),
            'truncated':     int(cl['truncated'].sum()),
            'avg latency, s':round(fc_['elapsed_s'].mean(), 2),
            'empty':         int(mv['commentary_empty'].sum()),
        })

    df_cmp = pd.DataFrame(records)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle('Trace File Comparison', fontsize=13, fontweight='bold')

    for ax, (col, title, color) in zip(axes, [
        ('avg prompt',    'Avg Prompt Length',   '#2196F3'),
        ('avg response',  'Avg Response Length', '#4C72B0'),
        ('avg thinking',  'Avg Thinking',         '#DD8452'),
        ('avg latency, s','Avg Latency (s)',       '#8172B2'),
    ]):
        ax.bar(df_cmp['file'], df_cmp[col], color=color, alpha=0.8, edgecolor='white')
        ax.set_title(title)
        ax.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.show()
    display(df_cmp)
else:
    print('Only one trace file found — comparison unavailable.')

## Anomaly Detector Flags
Traces which section flags fire on each game move. Data is collected in `eval_game.py` via the `trace["anomaly"]` field.

In [ ]:
# ── Anomaly detector flag data ────────────────────────────────────────────────
data = json.loads(TRACE_FILE.read_text())
FLAG_COLS = [
    'show_score_table', 'show_game_phase', 'show_pawn_structure',
    'show_space', 'show_mobility', 'show_makogonov',
]
FLAG_LABELS = {
    'show_score_table':    'Score table',
    'show_game_phase':     'Game phase',
    'show_pawn_structure': 'Pawn structure',
    'show_space':          'Space',
    'show_mobility':       'Mobility',
    'show_makogonov':      'Makogonov',
}

anm_rows = []
for t in data['traces']:
    if t.get('san') is None or 'anomaly' not in t:
        continue
    a = t['anomaly']
    row = {
        'index':       t['index'],
        'san':         t.get('san'),
        'move_number': t.get('move_number'),
        'color':       t.get('color'),
        'eval_cp':     t.get('eval_cp'),
        'eval_loss_cp': t.get('eval_loss_cp'),
        'game_phase':  a.get('game_phase', ''),
        'prev_phase':  a.get('prev_game_phase', ''),
        'phase_transition': a.get('phase_transition_remark', False),
        'anomaly_summary': a.get('anomaly_summary', False),
        **{f: a.get(f, False) for f in FLAG_COLS},
        'eval_delta_cp':       a.get('eval_delta_cp'),
        'max_pawn_weaknesses': a.get('max_pawn_weaknesses'),
        'space_diff':          a.get('space_diff'),
        'score_mobility_abs':  a.get('score_mobility_abs'),
    }
    for k, v in a.get('thresholds', {}).items():
        row[f'thr_{k}'] = v
    anm_rows.append(row)

anm = pd.DataFrame(anm_rows)
print(f'Positions with detector data: {len(anm)}')
if anm.empty:
    print('No data — run eval_game.py with the current version.')
else:
    print()
    print(f'  {"Flag":<24}  {"Fired":>6}  {"Off":>6}  {"Rate":>7}')
    print('  ' + '-' * 50)
    for f in FLAG_COLS:
        on  = int(anm[f].sum())
        off = len(anm) - on
        pct = on / len(anm) * 100
        print(f'  {FLAG_LABELS[f]:<24}  {on:>6}  {off:>6}  {pct:>6.1f}%')
    pt = int(anm['phase_transition'].sum())
    print(f'  {"Phase transition":<24}  {pt:>6}  {len(anm)-pt:>6}  {pt/len(anm)*100:>6.1f}%')
    print(f'  {"Structural alerts":<24}  {int(anm["anomaly_summary"].sum()):>6}')


In [ ]:
# ── Flag timeline and heatmap ─────────────────────────────────────────────────
if not anm.empty:
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    fig.suptitle('Anomaly Detector Flags per Move', fontsize=13, fontweight='bold')

    # ── Heatmap ──────────────────────────────────────────────────────────────
    ax = axes[0]
    hm_data = anm.set_index('index')[FLAG_COLS].T.astype(int)
    im = ax.imshow(hm_data.values, aspect='auto', cmap='RdYlGn',
                   vmin=0, vmax=1, interpolation='none')
    ax.set_yticks(range(len(FLAG_COLS)))
    ax.set_yticklabels([FLAG_LABELS[f] for f in FLAG_COLS])
    ax.set_xticks(range(len(anm)))
    ax.set_xticklabels(anm['san'].tolist(), rotation=90, fontsize=7)
    ax.set_title('Heatmap: green = on, red = off')
    plt.colorbar(im, ax=ax, fraction=0.015, pad=0.02)

    for _, row in anm.iterrows():
        if row['phase_transition']:
            xi = anm[anm['index'] == row['index']].index[0]
            ax.axvline(xi, color='blue', lw=1.5, linestyle='--', alpha=0.7)

    # ── Step plot per flag ────────────────────────────────────────────────────
    ax2 = axes[1]
    colors_f = plt.cm.Set1(np.linspace(0, 0.85, len(FLAG_COLS)))
    x_pos = list(range(len(anm)))
    for i, (f, c) in enumerate(zip(FLAG_COLS, colors_f)):
        y = anm[f].astype(int).values + i * 1.3
        ax2.step(x_pos, y, where='mid', color=c, lw=2, label=FLAG_LABELS[f])
        ax2.fill_between(x_pos, i * 1.3, y, step='mid', alpha=0.12, color=c)
    ax2.set_yticks([i * 1.3 + 0.5 for i in range(len(FLAG_COLS))])
    ax2.set_yticklabels([FLAG_LABELS[f] for f in FLAG_COLS])
    ax2.set_xticks(x_pos[::4])
    ax2.set_xticklabels(anm['san'].tolist()[::4], rotation=45, fontsize=8)
    ax2.set_title('Time series: 1 = on, 0 = off')
    ax2.set_xlabel('Move')

    for xi, row in enumerate(anm.itertuples()):
        if row.phase_transition:
            ax2.axvline(xi, color='blue', lw=1.5, linestyle='--', alpha=0.6,
                        label='Phase change' if xi == anm[anm['phase_transition']].index[0] else '')

    ax2.legend(fontsize=8, loc='upper right', ncol=3)
    plt.tight_layout()
    plt.show()

    print('\nGame phases:')
    for _, row in anm.drop_duplicates('game_phase', keep='first').iterrows():
        print(f'  Move {row["san"]} (#{row["index"]}): {row["game_phase"]}')


## Threshold Sensitivity Analysis
For each numeric gate: what percentage of moves would fire the flag at each threshold value.  
The red vertical line marks the current threshold from the configuration.

In [ ]:
# ── Threshold sensitivity sweep ───────────────────────────────────────────────
if not anm.empty:
    SWEEPS = [
        ('show_score_table',    'eval_delta_cp',       'score_jump_threshold_cp',
         np.arange(0, 301, 5),  'cp',          'Score table'),
        ('show_pawn_structure', 'max_pawn_weaknesses', 'pawn_weakness_threshold',
         np.arange(0, 12, 0.5), 'weaknesses',  'Pawn structure'),
        ('show_space',          'space_diff',          'space_imbalance_threshold',
         np.arange(0, 26, 0.5), 'squares',     'Space'),
        ('show_mobility',       'score_mobility_abs',  'mobility_score_threshold',
         np.arange(0, 251, 5),  'cp',           'Mobility'),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(
        'Threshold Sensitivity: % of moves with flag = True\n'
        '(red line = current config value)',
        fontsize=12, fontweight='bold'
    )
    axes = axes.flatten()

    for ax, (flag, input_col, thr_col, thresholds, unit, label) in zip(axes, SWEEPS):
        values = anm[input_col].dropna()
        if values.empty:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center', va='center')
            ax.set_title(label)
            continue

        pcts = [(values >= thr).mean() * 100 for thr in thresholds]

        ax.plot(thresholds, pcts, color='#2196F3', lw=2.5)
        ax.fill_between(thresholds, pcts, alpha=0.12, color='#2196F3')

        thr_key = f'thr_{thr_col}'
        current_thr = anm[thr_key].iloc[0] if thr_key in anm.columns else None
        if current_thr is not None:
            current_pct = (values >= current_thr).mean() * 100
            ax.axvline(current_thr, color='red', lw=2, linestyle='--',
                       label=f'Config: {current_thr} {unit} → {current_pct:.0f}%')
            ax.scatter([current_thr], [current_pct], color='red', s=80, zorder=5)

        for p, col, ls in [(25, '#aaa', ':'), (50, '#888', '--'), (75, '#666', ':')]:
            pv = float(np.percentile(values, p))
            ax.axvline(pv, color=col, lw=0.8, linestyle=ls, alpha=0.7)
            ax.text(pv, 102, f'p{p}', fontsize=7, color=col, ha='center')

        ax.set_title(label, fontsize=11)
        ax.set_xlabel(f'Threshold ({unit})')
        ax.set_ylabel('% moves with flag = True')
        ax.set_ylim(-2, 110)
        ax.axhline(50, color='gray', lw=0.5, linestyle=':', alpha=0.5)
        if current_thr is not None:
            ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    print(f'\n  {"Flag":<22}  {"Threshold key":>30}  {"Unit":>10}  {"Value":>8}  {"Fired":>8}')
    print('  ' + '-' * 85)
    for flag, input_col, thr_col, _, unit, label in SWEEPS:
        if input_col not in anm.columns:
            continue
        values = anm[input_col].dropna()
        thr_key = f'thr_{thr_col}'
        current_thr = anm[thr_key].iloc[0] if thr_key in anm.columns else None
        if current_thr is not None and len(values) > 0:
            fires = int((values >= current_thr).sum())
            total = len(values)
            print(f'  {label:<22}  {thr_col:>30}  {unit:>10}  {current_thr:>8}  '
                  f'{fires:>3}/{total} ({fires/total*100:.0f}%)')

In [ ]:
# ── Joint threshold matrix ────────────────────────────────────────────────────
# Shows % of moves where both flags fire simultaneously for every threshold pair.
# Useful for tuning two thresholds at once.
if not anm.empty:
    PAIRS = [
        ('eval_delta_cp',       'score_jump_threshold_cp',   'cp',     'Score table'),
        ('score_mobility_abs',  'mobility_score_threshold',  'cp',     'Mobility'),
    ]
    pair_data = [(c, t, u, l) for c, t, u, l in PAIRS if anm[c].notna().any()]

    if len(pair_data) >= 2:
        (c1, t1, u1, l1), (c2, t2, u2, l2) = pair_data[:2]
        v1 = anm[c1].dropna()
        v2 = anm[c2].dropna()
        grid1 = np.arange(0, max(v1) * 1.1 + 1, max(v1) / 20)
        grid2 = np.arange(0, max(v2) * 1.1 + 1, max(v2) / 20)

        matrix = np.zeros((len(grid2), len(grid1)))
        for i, thr2 in enumerate(grid2):
            for j, thr1 in enumerate(grid1):
                both = ((v1 >= thr1) & (v2 >= thr2)).mean() * 100
                matrix[i, j] = both

        fig, ax = plt.subplots(figsize=(10, 7))
        im = ax.imshow(matrix, aspect='auto', origin='lower', cmap='YlOrRd',
                       extent=[grid1[0], grid1[-1], grid2[0], grid2[-1]])
        plt.colorbar(im, ax=ax, label='% moves where both flags fire')

        curr1 = anm[f'thr_{t1}'].iloc[0] if f'thr_{t1}' in anm.columns else None
        curr2 = anm[f'thr_{t2}'].iloc[0] if f'thr_{t2}' in anm.columns else None
        if curr1:
            ax.axvline(curr1, color='white', lw=2, linestyle='--', label=f'{l1} thr={curr1}')
        if curr2:
            ax.axhline(curr2, color='cyan',  lw=2, linestyle='--', label=f'{l2} thr={curr2}')
        if curr1 and curr2:
            both_val = ((v1 >= curr1) & (v2 >= curr2)).mean() * 100
            ax.scatter([curr1], [curr2], color='white', s=120, zorder=5)
            ax.text(curr1, curr2 + max(grid2) * 0.04,
                    f'{both_val:.0f}%', color='white', fontsize=10,
                    ha='center', fontweight='bold')

        ax.set_xlabel(f'{l1} threshold ({u1})')
        ax.set_ylabel(f'{l2} threshold ({u2})')
        ax.set_title(f'Joint firing rate: {l1} + {l2}', fontsize=12)
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()
    else:
        print('Insufficient numeric data for joint matrix.')